In [ ]:
import numpy as np
from astropy.table import Table
from scipy.interpolate import interp1d


In [ ]:
import filter_utils
import ana_utils

In [ ]:
import matplotlib.pyplot as plt
import arya

In [ ]:
import imf

In [ ]:
def get_default_params(objname):
    obs_props = ana_utils.get_obs_props(objname)
    return filter_utils.filter_params(
        dm = obs_props["dm"], 
        iso_fe_h = obs_props["iso_fe_h"],
        iso_log_age = np.log10(obs_props["iso_age"]),
        objname = objname, 
        mode="gri", 
        A_V = 0,
        r23_max_sigma=None,
        r_cen = 40/60,
        iso_width=0.15
    )
    

In [ ]:
def sample_kroupa_imf(nstars, mmin, mmax):
    """
    Sample stellar masses from a Kroupa (2001) IMF
    between mmin and mmax.
    """
    # Define slopes
    alpha1 = 1.3
    alpha2 = 2.3
    m_break = 0.5

    # Power-law integral
    def integral(m, alpha):
        if alpha == 1:
            return np.log(m)
        return m**(1 - alpha) / (1 - alpha)

    # Compute normalization pieces
    if mmax <= m_break:
        # Single slope region
        A = integral(mmax, alpha1) - integral(mmin, alpha1)
        u = np.random.uniform(size=nstars)
        return ((u * A + integral(mmin, alpha1)) * (1 - alpha1))**(1/(1-alpha1))
    
    # Two-segment IMF
    I1 = integral(m_break, alpha1) - integral(mmin, alpha1)
    I2 = integral(mmax, alpha2) - integral(m_break, alpha2)
    Itot = I1 + I2

    u = np.random.uniform(size=nstars) * Itot

    masses = np.zeros(nstars)

    mask1 = u < I1
    mask2 = ~mask1

    # Low-mass branch
    masses[mask1] = (
        (u[mask1] + integral(mmin, alpha1)) * (1 - alpha1)
    )**(1/(1-alpha1))

    # High-mass branch
    masses[mask2] = (
        (u[mask2] - I1 + integral(m_break, alpha2)) * (1 - alpha2)
    )**(1/(1-alpha2))

    return masses


In [ ]:
def sample_population_from_isochrone(
    iso,
    M,
    sigma_g,
    sigma_r,
    sigma_i
):
    """
    Sample a stellar population from a MIST isochrone.
    
    Parameters
    ----------
    iso : astropy.table.Table
        Must contain columns:
        'initial_mass', 'g', 'r', 'i'
    nstars : int
        Number of stars to sample
    sigma_g, sigma_r, sigma_i : functions
        Functions returning magnitude uncertainty
        given a magnitude.
    """

    # Sort isochrone by mass
    iso = iso[np.argsort(iso["initial_mass"])]

    mass_iso = iso["initial_mass"]
    g_iso = iso["g"]
    r_iso = iso["r"]
    i_iso = iso["i"]

    # Define allowed mass range
    mmin = mass_iso.min()
    mmax = mass_iso.max()

    # Sample masses from IMF
    masses = imf.make_cluster(M)

    # Interpolate magnitudes as function of mass
    g_interp = interp1d(mass_iso, g_iso, bounds_error=False, fill_value=np.nan)
    r_interp = interp1d(mass_iso, r_iso, bounds_error=False, fill_value=np.nan)
    i_interp = interp1d(mass_iso, i_iso, bounds_error=False, fill_value=np.nan)

    g_true = g_interp(masses)
    r_true = r_interp(masses)
    i_true = i_interp(masses)

    # Remove stars outside interpolation range
    valid = np.isfinite(g_true)
    masses = masses[valid]
    g_true = g_true[valid]
    r_true = r_true[valid]
    i_true = i_true[valid]

    # Apply photometric uncertainties
    g_obs = g_true + np.random.normal(0, sigma_g(g_true))
    r_obs = r_true + np.random.normal(0, sigma_r(r_true))
    i_obs = i_true + np.random.normal(0, sigma_i(i_true))

    # Return as astropy Table
    return Table({
        "mass": masses,
        "g_true": g_true,
        "r_true": r_true,
        "i_true": i_true,
        "g_obs": g_obs,
        "r_obs": r_obs,
        "i_obs": i_obs,
        "g-r": g_obs - r_obs,
        "r-i": r_obs - i_obs
    })

# Main drag

In [ ]:
cat = ana_utils.read_catalogue("yasone1", filter_bad=False, catname="allcolours", shiftname="allcolours_panstarrs_shift", deredden=True)


In [ ]:
err_g = ana_utils.fit_err(cat["G_MAG"], cat["G_MAG_ERR"])
err_r = ana_utils.fit_err(cat["R_MAG"], cat["R_MAG_ERR"])
err_i = ana_utils.fit_err(cat["I_MAG"], cat["I_MAG_ERR"])

gr_err = filter_utils.get_gr_err(cat, params)
ri_err = filter_utils.get_ri_err(cat, params)

In [ ]:
Ms = sample_kroupa_imf(100_000, 0.08, 100)
Ms2 = imf.make_cluster(500.0)

In [ ]:
plt.hist(Ms)
plt.s

In [ ]:
params = get_default_params("yasone1")

In [ ]:
params.iso_width=0.15

In [ ]:
iso = filter_utils.isochrones[params.iso_fe_h][params.iso_log_age]
dm = params.dm

iso["g"] = iso["SDSS_g"] + dm
iso["r"] = iso["SDSS_r"] + dm
iso["i"] = iso["SDSS_i"] + dm

In [ ]:
pop = sample_population_from_isochrone(
    iso,
    25.0,
    err_g,
    err_r,
    err_i,
)


In [ ]:
np.sum(pop["r_obs"] < 26)

In [ ]:
for i in range(10):
    pop = sample_population_from_isochrone(
        iso,
        25.0,
        err_g,
        err_r,
        err_i,
    )
    plot_cluster(pop, params)
    print(np.sum(pop["r_obs"] < 26))

In [ ]:
def plot_cluster(pop, params):
    fig, axs = plt.subplots(1, 2, figsize=(5, 2))
    
    plt.sca(axs[1])
    plt.scatter(pop["r_obs"] - pop["i_obs"], pop["r_obs"])
    filter_utils.plot_iso_ri(params, ri_err)
    plt.ylim(26, 16)
    plt.xlim(-1, 2)
    plt.xlabel("r - i")
    plt.ylabel("r")
    
    
    plt.sca(axs[0])
    
    plt.scatter(pop["g_obs"] - pop["r_obs"], pop["g_obs"])
    filter_utils.plot_iso_gr(params, gr_err)
    
    plt.ylim(27, 16)
    plt.xlim(-1, 3)
    
    plt.xlabel("g - r")
    plt.ylabel("g")
    plt.tight_layout()

In [ ]:
plt.hist(np.log10(Ms), density=True)
plt.hist(np.log10(Ms2), density=True, histtype="step")
plt.yscale("log")